# SLM 학습 & 튜닝 (emotion-bot)

대화/감정 분석용 SLM fine-tuning 템플릿.

- 로그: `experiments/slm_runs.jsonl`
- 설정: `configs/slm/example.yaml`
- 데이터: `data/slm/train.jsonl` (한 줄당 JSON)

추가 설치: `pip install transformers datasets accelerate`

In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

ROOT = Path.cwd().resolve()
for _ in range(5):
    if (ROOT / "config.py").is_file():
        break
    if ROOT.parent == ROOT:
        break
    ROOT = ROOT.parent
else:
    raise FileNotFoundError("emotion-bot 루트를 찾지 못했습니다.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

EXPERIMENTS_LOG = ROOT / "experiments" / "slm_runs.jsonl"
EXPERIMENTS_LOG.parent.mkdir(parents=True, exist_ok=True)

def log_run(params, metrics, notes=""):
    record = {
        "run_id": datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
        "params": params,
        "metrics": metrics,
        "notes": notes,
    }
    with EXPERIMENTS_LOG.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return record

def load_log():
    if not EXPERIMENTS_LOG.is_file() or EXPERIMENTS_LOG.stat().st_size == 0:
        return pd.DataFrame()
    rows = [json.loads(ln) for ln in EXPERIMENTS_LOG.read_text(encoding="utf-8").splitlines() if ln.strip()]
    flat = []
    for r in rows:
        row = {"run_id": r["run_id"]}
        row.update({f"p_{k}": v for k, v in r.get("params", {}).items()})
        row.update({f"m_{k}": v for k, v in r.get("metrics", {}).items()})
        flat.append(row)
    return pd.DataFrame(flat)

print("ROOT:", ROOT)

## 1. 파라미터

In [ ]:
YAML_CFG = ROOT / "configs" / "slm" / "example.yaml"
cfg = yaml.safe_load(YAML_CFG.read_text(encoding="utf-8")) if YAML_CFG.is_file() else {}
TRAIN_PARAMS = {
    "run_name": cfg.get("run_name", "slm_nb_001"),
    "model_name": cfg.get("model_name", "meta-llama/Llama-3.2-1B-Instruct"),
    "epochs": cfg.get("epochs", 3),
    "batch_size": cfg.get("batch_size", 4),
    "lr": cfg.get("lr", 2e-5),
    "max_seq_len": cfg.get("max_seq_len", 512),
}
DATA_PATH = Path(cfg.get("data_path", "data/slm/train.jsonl"))
print("data:", DATA_PATH, DATA_PATH.is_file())
TRAIN_PARAMS

## 2. 학습 (템플릿)

`RUN_TRAINING=True` 후 HuggingFace Trainer 등 프로젝트에 맞게 완성하세요.

In [ ]:
RUN_TRAINING = False

if not DATA_PATH.is_file():
    print(f"데이터 없음: {DATA_PATH}")
elif RUN_TRAINING:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from datasets import load_dataset

    ds = load_dataset("json", data_files=str(DATA_PATH), split="train")
    tokenizer = AutoTokenizer.from_pretrained(TRAIN_PARAMS["model_name"])
    model = AutoModelForCausalLM.from_pretrained(TRAIN_PARAMS["model_name"])
    # TODO: tokenize + Trainer(...).train()
    metrics = {"samples": len(ds)}
    log_run(TRAIN_PARAMS, metrics, notes="slm template")
else:
    print("RUN_TRAINING=False — 설정만 확인")

## 3. run 비교

In [ ]:
df = load_log()
display(df if not df.empty else "experiments/slm_runs.jsonl 비어 있음")